# 추가실험 eval — **노드 2/4** (이 서버 GPU 2개)

4개 서버로 나눠 도는 노트북 중 **2번**. 서버마다 `exp_eval_node1~4` 중 하나를 연다.
20개 eval(실험1 12 + 실험2 8)을 4노드 round-robin 으로 나눔 → 노드당 5개, 노드당 2 GPU = **총 8 동시**.

**재학습 없음** — 전부 기존 체크포인트 재사용(overlap=추론전용 재해석, TE=eval-time). 위→아래 실행.


## 0) 부팅 + 이 노드 담당 목록


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

TASK  = 'libero_10'
SEEDS = [0, 1]                       # 추가실험 = 2 seed
N_EP  = 100                          # task당 100 에피소드 (LIBERO-10 → overall 1000). 메인은 500.
EXP1 = ['act', 'act_te', 'act_overlap', 'bimamba', 'bimamba_te', 'bimamba_mosaic']
EXP2 = ['acm2', 'acm2_carry', 'acm2_overlap', 'acm2_mosaic']
UNITS = [(t, s) for t in EXP1 + EXP2 for s in SEEDS]      # 20 = (6+4)tag x 2seed

NODE_IDX, NNODES = 1, 4          # ← 이 노트북 = 노드 2/4 (서버마다 파일만 다름)
MY   = UNITS[NODE_IDX::NNODES]       # 이 노드 담당 (round-robin)
GPUS = v23.available_gpus()          # 이 노드 GPU 2개 → 2 run 동시

print(f'노드 {NODE_IDX+1}/{NNODES} | GPU {GPUS} | 담당 {len(MY)} run')
for t, s in MY:
    print(f'   {t:16} seed{s}  ({v23.MODEL_LABELS.get(t, t)})')

## 1) 상태 점검 (담당분 소스가 이 서버에 있나)


In [ ]:
# 담당분 소스 체크포인트/eval 상태 (재해석·TE 는 소스 태그로 판정)
def resolve(t, s):
    if t in cf.REINTERP:
        src, cd = cf._first_source(cf.REINTERP[t][0], s, TASK, cf.CKPT_STEP)
        return src, cd is not None
    if t.endswith('_te'):
        src = t[:-3]; return src, v23.best_ckpt_dir(src, s, TASK, how=cf.CKPT_STEP) is not None
    return t, v23.best_ckpt_dir(t, s, TASK, how=cf.CKPT_STEP) is not None

_MINEP = 10 * N_EP // 2              # 유효 eval: overall n_ep >= 500 (100ep×10÷2)
def done(t, s):
    info = cf.eval_rep_dir(t, s, TASK, 0, cf.CKPT_STEP) / 'eval_info.json'
    if not info.exists(): return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

print(f'{"tag":16}{"seed":5}{"소스":16}{"소스ckpt":9}{"eval":6}')
miss = []
for t, s in MY:
    src, has = resolve(t, s)
    print(f'{t:16}{s:<5}{str(src):16}{("있음" if has else "없음"):9}{("완료" if done(t, s) else "—"):6}')
    if not has: miss.append((t, s, src))
print('\n⚠️ 소스 없음:', miss, '→ mosaic_infer/acm2/act/bimamba 가 이 서버에 있어야 함') if miss \
    else print('\n✅ 담당분 소스 전부 존재 → 재학습 없이 eval')

## 2) overlap 재해석 (담당분만)


In [ ]:
# 담당분 overlap 재해석 (자기 몫만 → 노드 간 파일 경쟁 없음). *_te·직접 태그는 아무것도 안 함.
#   acm2_carry ← mosaic_infer(overlap 0=순수 carry) / acm2_mosaic ← mosaic_infer(overlap 10=MOSAIC)
cf.prepare_overlap_for(MY, TASK)

## 3) eval 실행


In [ ]:
# eval — 담당분을 GPU 2개로(2 동시). 1 rep x 100ep x 10 task. 끝난 run 은 자동 skip, 끊기면 이어서.
cf.run_libero_eval_jobs(MY, GPUS, task=TASK, n_episodes=N_EP, reps=[0])